# 环节 01 · 规格与任务族（配套 Notebook）
> 经典开源账：MiniMax-H3 Base。总揽：[环节00](./环节00-总揽与环节导航.md)

> 配套：[环节01-规格与任务族详解.md](./环节01-规格与任务族详解.md) ｜ 总揽：[环节00](./环节00-总揽与环节导航.md)
> 纯 Python，零依赖。验证：时长/画幅如何变成「合同约束」。


In [ ]:
SPECS = {
    "duration_s": (4, 15),
    "fps": 24,
    "short_side": 768,
    "audio_hz": 32000,
}

ASPECTS = {
    "16:9": (16, 9),
    "9:16": (9, 16),
    "1:1": (1, 1),
    "21:9": (21, 9),
}


def canvas(aspect, short=768):
    a, b = aspect
    if a >= b:
        h = short
        w = round(short * a / b)
    else:
        w = short
        h = round(short * b / a)
    # 必须能被 32 整除才进得了 32× 空间下采样（环节03）
    w = w - (w % 32)
    h = h - (h % 32)
    return w, h


print("画幅合同（短边768，对齐32）：")
for name, asp in ASPECTS.items():
    print(f"  {name:>5}  {canvas(asp)[0]}×{canvas(asp)[1]}")

print("\n时长合同：帧数必须能被时间压缩4整除")
for sec in range(4, 16):
    frames = sec * 24
    ok = frames % 4 == 0
    print(f"  {sec:2d}s → {frames:3d} 帧  {'OK' if ok else '不能被4整除'}")


In [ ]:
def pick_variant(n_images, n_videos, n_audios, has_text=True):
    files = n_images + n_videos + n_audios
    if n_videos or n_audios or n_images > 2:
        if n_audios and not (n_images or n_videos):
            return "非法：音频不能单独作为输入（开源日新闻；现网 HF 规格表未写）"
        if n_images > 9 or n_videos > 3 or n_audios > 3 or files > 12:
            return "超出 Ref2VA 上限"
        return "Ref2VA"
    if n_images <= 2 and has_text:
        return "FL2VA（0图=T2VA，1图=首或尾，2图=首尾）"
    return "无法路由"


cases = [
    ("纯文生", 0, 0, 0),
    ("首尾帧", 2, 0, 0),
    ("希区柯克+人+歌", 1, 1, 1),
    ("只有一段音频", 0, 0, 1),
    ("10张参考图", 10, 0, 0),
]
for name, i, v, a in cases:
    print(f"{name:12} → {pick_variant(i, v, a)}")
